# 🏠 California House Price Prediction with Neural Networks

This notebook demonstrates how to build a **neural network** to predict house prices using the California Housing dataset.

## What we'll learn:
- **Data loading and preprocessing** with scikit-learn
- **Building a neural network** with PyTorch
- **Training with regularization** (Dropout + L2)
- **Evaluating model performance** on test data
- **Visualizing results** and understanding parameter effects

---

In [ ]:
!pip install torch scikit-learn

In [ ]:
# --- Tools we need for our exercise---
import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Additional imports for visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading & Exploration

The **California Housing dataset** contains information about houses in California districts. Each sample has 8 features describing the district and the median house price.

In [ ]:
# 1) LOAD THE DATA
data     = fetch_california_housing()
features = data.data  # 8 input numbers for each house
prices   = data.target.reshape(-1, 1)  # Reshape per lo scaler

print(f"Number of samples: {len(prices)}")
print(f"Number of features: {features.shape[1]}")
print(f"Feature names: {data.feature_names}")

In [ ]:
# --- Data Visualization ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of house prices
axes[0, 0].hist(prices, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[0, 0].set_xlabel('House Price (in $100k)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('📊 Distribution of House Prices')
axes[0, 0].axvline(np.mean(prices), color='red', linestyle='--', label=f'Mean: ${np.mean(prices)*100:.0f}k')
axes[0, 0].legend()

# 2. Feature correlation with price
correlations = [np.corrcoef(features[:, i], prices.flatten())[0, 1] for i in range(features.shape[1])]
colors = ['green' if c > 0 else 'red' for c in correlations]
axes[0, 1].barh(data.feature_names, correlations, color=colors, alpha=0.7)
axes[0, 1].set_xlabel('Correlation with Price')
axes[0, 1].set_title('📈 Feature Correlation with House Price')
axes[0, 1].axvline(0, color='black', linewidth=0.5)

# 3. Median Income vs Price (strongest correlation)
axes[1, 0].scatter(features[:, 0], prices, alpha=0.3, s=5, c='steelblue')
axes[1, 0].set_xlabel('Median Income')
axes[1, 0].set_ylabel('House Price (in $100k)')
axes[1, 0].set_title('💰 Median Income vs House Price')

# 4. Geographic distribution
scatter = axes[1, 1].scatter(features[:, 7], features[:, 6], c=prices.flatten(), 
                              cmap='viridis', alpha=0.5, s=3)
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
axes[1, 1].set_title('🗺️ Geographic Distribution of Prices')
plt.colorbar(scatter, ax=axes[1, 1], label='Price ($100k)')

plt.tight_layout()
plt.show()

## 2. Data Preparation

We split the data into three sets:
- **Training set (70%)**: The model learns from this data
- **Validation set (15%)**: Used to monitor training and prevent overfitting  
- **Test set (15%)**: Final evaluation on unseen data

In [ ]:
# 2) SPLIT THE DATA (aggiungiamo anche validation set)

# We divide the dataset into 3 parts:
# training (learn), validation (check progress), test (final evaluation)

train_features, temp_features, train_prices, temp_prices = train_test_split(
    features, prices, test_size=0.3, random_state=42
)

val_features, test_features, val_prices, test_prices = train_test_split(
    temp_features, temp_prices, test_size=0.5, random_state=42
)

# Show how many samples are in each group
print("Training samples:", len(train_prices))
print("Validation samples:", len(val_prices))
print("Test samples:", len(test_prices))


In [ ]:
# 3) SCALE THE DATA

# We rescale the data so all numbers are comparable
# (mean = 0, standard deviation = 1)

feature_scaler = StandardScaler()
price_scaler = StandardScaler()

# Scale input data (house features)
train_features = feature_scaler.fit_transform(train_features)
val_features   = feature_scaler.transform(val_features)
test_features  = feature_scaler.transform(test_features)

# Scale output data (house prices)
train_prices = price_scaler.fit_transform(train_prices)
val_prices   = price_scaler.transform(val_prices)
test_prices  = price_scaler.transform(test_prices)

print("Data scaled: all values now have similar size")


### Why do we scale the data?

**Standardization** (mean=0, std=1) helps neural networks:
- Learn faster (similar scale for all features)
- Avoid numerical instability
- Treat all features equally initially

In [ ]:
# 4) CONVERT TO PYTORCH TENSORS
train_features = torch.tensor(train_features, dtype=torch.float32)
train_prices   = torch.tensor(train_prices, dtype=torch.float32)

val_features = torch.tensor(val_features, dtype=torch.float32)
val_prices = torch.tensor(val_prices, dtype=torch.float32)

test_features  = torch.tensor(test_features, dtype=torch.float32)
test_prices = torch.tensor(test_prices, dtype=torch.float32)


## 3. Neural Network Architecture

Our model uses:
- **6 layers** with decreasing neurons (256 → 128 → 64 → 32 → 16 → 1)
- **ReLU activation** for non-linearity
- **Dropout regularization** to prevent overfitting

```
Input (8 features) → 256 → 128 → 64 → 32 → 16 → 1 (price prediction)
```

In [ ]:
# 5) MODELLO CON DROPOUT PER REGOLARIZZAZIONE
class HousePriceModel(nn.Module):
    def __init__(self, n_features, dropout_rate=0.1):
        super().__init__()
        
        # Modello con dropout per regolarizzazione
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),    # First layer from input features to 256 neurons
            nn.ReLU(),                     # Non-linear activation
            nn.Dropout(dropout_rate),      # Dropout after first hidden layer

            nn.Linear(256, 128),           # Second  layer
            nn.ReLU(),                     # Non-linear activation
            nn.Dropout(dropout_rate),      # Dropout after second  layer

            nn.Linear(128, 64),            # Third  layer
            nn.ReLU(),
            nn.Dropout(dropout_rate),      # Dropout after third layer

            nn.Linear(64, 32),             # Fourth layer
            nn.ReLU(),                     # Non-linear activation

            nn.Linear(32, 16),             # Fifth layer
            nn.ReLU(),                     # Non-linear activation

            nn.Linear(16, 1)               # Output layer (no dropout here)
        )

    def forward(self, x):
        return self.network(x)


model = HousePriceModel(train_features.shape[1], dropout_rate=0.1)
print(model)


## 4. Training the Model

**Key hyperparameters:**
- `learning_rate`: How big each learning step is
- `weight_decay`: L2 regularization strength
- `dropout_rate`: Fraction of neurons to disable during training
- `epochs`: Number of training iterations

In [ ]:
# We measure how wrong the predictions are
loss_function = nn.MSELoss()

# How big each learning step is
learning_rate = 0.005

# Weight decay (L2 regularization) - penalizes large weights
weight_decay = 1e-4

# Tool that updates the model to make predictions better
# Added weight_decay for L2 regularization
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# How many times we repeat the learning process
number_of_epochs = 2000

# Track training history for visualization
train_losses = []
val_losses = []

for epoch in range(number_of_epochs):

    # --- TRAINING ---
    model.train()                         # learning mode (dropout active)
    predicted_prices = model(train_features)
    train_loss = loss_function(predicted_prices, train_prices)

    optimizer.zero_grad()                 # reset old gradients
    train_loss.backward()                 # understand mistakes
    optimizer.step()                      # improve the model

    # --- VALIDATION ---
    model.eval()                          # evaluation mode (dropout disabled)
    with torch.no_grad():
        val_predictions = model(val_features)
        val_loss = loss_function(val_predictions, val_prices)

    # Store losses for plotting
    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    # Print progress sometimes
    if epoch % 50 == 0:
        print(
            f"Epoch {epoch} | "
            f"Train error: {train_loss.item():.4f} | "
            f"Validation error: {val_loss.item():.4f}"
        )

In [ ]:
# --- Training History Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Training Loss', alpha=0.8)
axes[0].plot(val_losses, label='Validation Loss', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('📉 Training & Validation Loss Over Time')
axes[0].legend()
axes[0].set_yscale('log')

# Zoomed view (last 500 epochs)
axes[1].plot(train_losses[-500:], label='Training Loss', alpha=0.8)
axes[1].plot(val_losses[-500:], label='Validation Loss', alpha=0.8)
axes[1].set_xlabel('Epoch (last 500)')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('🔍 Loss Convergence (Last 500 Epochs)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Final Training Loss: {train_losses[-1]:.4f}")
print(f"✅ Final Validation Loss: {val_losses[-1]:.4f}")

## 5. Model Evaluation

Let's evaluate our model on the **test set** (data it has never seen during training).

In [ ]:
# 7) EVALUATE ON TEST SET

# Switch the model to evaluation mode
model.eval()
with torch.no_grad():
    test_predictions_scaled = model(test_features).cpu().numpy()

# De-scale predictions and targets to original units (prices in $100k)
test_predictions = price_scaler.inverse_transform(test_predictions_scaled)
test_prices_original = price_scaler.inverse_transform(test_prices.cpu().numpy())

# Calculate metrics on original values
mae = np.mean(np.abs(test_predictions - test_prices_original))
rmse = np.sqrt(np.mean((test_predictions - test_prices_original) ** 2))
mape = np.mean(np.abs((test_predictions - test_prices_original) / test_prices_original)) * 100

# Calculate R² score
ss_res = np.sum((test_prices_original - test_predictions) ** 2)
ss_tot = np.sum((test_prices_original - np.mean(test_prices_original)) ** 2)
r2_score = 1 - (ss_res / ss_tot)

# Display results in a clear format
print("=" * 60)
print("                    FINAL RESULTS")
print("=" * 60)
print()
print("Model Performance Metrics:")
print("-" * 40)
print(f"   Mean Absolute Error:     ${mae * 100000:>12,.0f}")
print(f"   Mean Absolute % Error:        {mape:>12.2f}%")

In [ ]:
# --- Prediction Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predicted vs Actual scatter plot
axes[0].scatter(test_prices_original, test_predictions, alpha=0.5, s=10)
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($100k)')
axes[0].set_ylabel('Predicted Price ($100k)')
axes[0].set_title('🎯 Predicted vs Actual Prices')
axes[0].legend()
axes[0].set_xlim(0, 5.5)
axes[0].set_ylim(0, 5.5)

# 2. Residuals distribution
residuals = test_predictions.flatten() - test_prices_original.flatten()
axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error ($100k)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('📊 Distribution of Prediction Errors')

# 3. Sample predictions comparison
sample_idx = np.random.choice(len(test_prices_original), 20, replace=False)
x_pos = np.arange(20)
width = 0.35
axes[2].bar(x_pos - width/2, test_prices_original[sample_idx].flatten(), width, label='Actual', alpha=0.8)
axes[2].bar(x_pos + width/2, test_predictions[sample_idx].flatten(), width, label='Predicted', alpha=0.8)
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Price ($100k)')
axes[2].set_title('🏠 Sample Predictions vs Actual')
axes[2].legend()
axes[2].set_xticks(x_pos)

plt.tight_layout()
plt.show()

---

## 6. 🔬 Hyperparameter Analysis

Let's explore how different **learning rates** affect the training process. This helps understand why hyperparameter tuning is crucial.

In [ ]:
# --- Experiment: Effect of Learning Rate ---

def train_with_lr(learning_rate, epochs=500):
    """Train a model with a specific learning rate and return loss history."""
    model_exp = HousePriceModel(train_features.shape[1], dropout_rate=0.1)
    optimizer_exp = torch.optim.Adam(model_exp.parameters(), lr=learning_rate, weight_decay=1e-4)
    
    train_history = []
    val_history = []
    
    for epoch in range(epochs):
        model_exp.train()
        pred = model_exp(train_features)
        loss = loss_function(pred, train_prices)
        
        optimizer_exp.zero_grad()
        loss.backward()
        optimizer_exp.step()
        
        model_exp.eval()
        with torch.no_grad():
            val_pred = model_exp(val_features)
            val_loss = loss_function(val_pred, val_prices)
        
        train_history.append(loss.item())
        val_history.append(val_loss.item())
    
    return train_history, val_history

# Test different learning rates
learning_rates = [0.0001, 0.001, 0.005, 0.01, 0.05]
results = {}

print("🔄 Training models with different learning rates...")
for lr in learning_rates:
    print(f"  Training with lr={lr}...")
    results[lr] = train_with_lr(lr, epochs=500)
print("✅ Done!")

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr in learning_rates:
    axes[0].plot(results[lr][0], label=f'lr={lr}', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('📈 Training Loss vs Learning Rate')
axes[0].legend()
axes[0].set_yscale('log')

for lr in learning_rates:
    axes[1].plot(results[lr][1], label=f'lr={lr}', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('📈 Validation Loss vs Learning Rate')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# Show final losses
print("\n📊 Final Validation Losses:")
for lr in learning_rates:
    print(f"   lr={lr}: {results[lr][1][-1]:.4f}")

### 📊 Key Observations:

| Learning Rate | Effect |
|--------------|--------|
| **Too small** (0.0001) | Slow convergence, may not reach optimal |
| **Good range** (0.001-0.01) | Stable learning, good final performance |
| **Too large** (0.05+) | Unstable training, may overshoot minimum |

---

### Effect of Dropout Rate

Dropout helps prevent **overfitting** by randomly disabling neurons during training.

In [ ]:
# --- Experiment: Effect of Dropout Rate ---

def train_with_dropout(dropout_rate, epochs=500):
    """Train a model with a specific dropout rate and return loss history."""
    model_exp = HousePriceModel(train_features.shape[1], dropout_rate=dropout_rate)
    optimizer_exp = torch.optim.Adam(model_exp.parameters(), lr=0.005, weight_decay=1e-4)
    
    train_history = []
    val_history = []
    
    for epoch in range(epochs):
        model_exp.train()
        pred = model_exp(train_features)
        loss = loss_function(pred, train_prices)
        
        optimizer_exp.zero_grad()
        loss.backward()
        optimizer_exp.step()
        
        model_exp.eval()
        with torch.no_grad():
            val_pred = model_exp(val_features)
            val_loss = loss_function(val_pred, val_prices)
        
        train_history.append(loss.item())
        val_history.append(val_loss.item())
    
    return train_history, val_history

# Test different dropout rates
dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.5]
dropout_results = {}

print("🔄 Training models with different dropout rates...")
for dr in dropout_rates:
    print(f"  Training with dropout={dr}...")
    dropout_results[dr] = train_with_dropout(dr, epochs=500)
print("✅ Done!")

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for dr in dropout_rates:
    axes[0].plot(dropout_results[dr][0], label=f'dropout={dr}', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('🎲 Training Loss vs Dropout Rate')
axes[0].legend()

for dr in dropout_rates:
    axes[1].plot(dropout_results[dr][1], label=f'dropout={dr}', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('🎲 Validation Loss vs Dropout Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

# Show gap between train and validation (overfitting indicator)
print("\n📊 Train-Validation Gap (lower = less overfitting):")
for dr in dropout_rates:
    gap = dropout_results[dr][0][-1] - dropout_results[dr][1][-1]
    print(f"   dropout={dr}: gap = {abs(gap):.4f}")

---

## 🎓 Summary

In this notebook we learned:

1. **Data preprocessing** is crucial - scaling helps neural networks learn effectively
2. **Network architecture** matters - we used a deep network with multiple hidden layers
3. **Regularization techniques** (Dropout + L2) help prevent overfitting
4. **Hyperparameter tuning** significantly affects model performance:
   - Learning rate controls convergence speed and stability
   - Dropout rate balances between underfitting and overfitting

### 💡 Tips for Improvement:
- Try different architectures (more/fewer layers, different sizes)
- Experiment with batch training for larger datasets
- Consider learning rate scheduling
- Use early stopping to prevent overfitting